<a href="https://colab.research.google.com/github/Jitendra0410/Twitter-Sentiment-Analysis/blob/main/DAV_Twitter_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 📌 Step 1: Install & import necessary libraries
!pip install -q scikit-learn pandas plotly seaborn matplotlib

import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder


In [ ]:

# 📌 Step 2: Load datasets (files already in Colab runtime)

cols = ['tweet_id', 'entity', 'sentiment', 'content']

# 📌 Step 2: Load datasets (files already in Colab)

cols = ['tweet_id', 'entity', 'sentiment', 'content']
train_df = pd.read_csv('twitter_training.csv', names=cols)
val_df = pd.read_csv('twitter_validation.csv', names=cols)

# Drop missing
train_df.dropna(subset=['content'], inplace=True)
val_df.dropna(subset=['content'], inplace=True)

# Sample for speed with safe check
train_df = train_df.sample(min(10000, len(train_df)), random_state=42)
val_df = val_df.sample(min(2000, len(val_df)), random_state=42)

# Encode labels
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['sentiment'])
val_df['label'] = le.transform(val_df['sentiment'])
val_df = pd.read_csv('twitter_validation.csv', names=cols)

# Drop missing
train_df.dropna(subset=['content'], inplace=True)
val_df.dropna(subset=['content'], inplace=True)

# Sample for speed
train_df = train_df.sample(min(10000, len(train_df)), random_state=42)
val_df = val_df.sample(min(2000, len(val_df)), random_state=42)


# Encode labels
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['sentiment'])
val_df['label'] = le.transform(val_df['sentiment'])

print("Train class distribution:\n", train_df['sentiment'].value_counts())


Train class distribution:
 sentiment
Negative      2936
Positive      2780
Neutral       2414
Irrelevant    1870
Name: count, dtype: int64


In [ ]:

# 📌 Step 3: Train a Naive Bayes model
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', MultinomialNB())
])
pipeline.fit(train_df['content'], train_df['label'])
val_preds = pipeline.predict(val_df['content'])


In [ ]:

# 📊 Step 4: Plotly Visualizations

# Sentiment Distribution
fig = px.pie(train_df, names='sentiment', title='Sentiment Distribution in Training Data')
fig.show()

# Prediction Results
train_df['text_length'] = train_df['content'].apply(len)
avg_len_df = train_df.groupby('sentiment')['text_length'].mean().reset_index()
fig1 = px.bar(avg_len_df, x='sentiment', y='text_length', title='Average Tweet Length by Sentiment')
fig1.show()

fig0 = px.box(train_df, x='sentiment', y='text_length', title='Tweet Length Distribution by Sentiment')
fig0.show()
fig4 = px.histogram(train_df, x='text_length', nbins=30, title='Distribution of Tweet Lengths')
fig4.show()


# Confusion Matrix (Heatmap)
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_df['label'], val_preds)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
fig7 = px.imshow(cm_df, text_auto=True, title='Confusion Matrix Heatmap')
fig7.show()



In [ ]:
# Convert classification report to DataFrame
from sklearn.metrics import classification_report
import pandas as pd
import plotly.express as px

print("📋 Classification Report:")
print(classification_report(val_df['label'], val_preds, target_names=le.classes_))

report_dict = classification_report(val_df['label'], val_preds, target_names=le.classes_, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().reset_index()

# Keep only actual classes, drop avg/accuracy
report_df = report_df[report_df['index'].isin(le.classes_)]

# Melt for bar plot (wide to long format)
melted = report_df.melt(id_vars='index', value_vars=['precision', 'recall', 'f1-score'],
                        var_name='Metric', value_name='Score')

# Plotly grouped bar chart
fig = px.bar(melted, x='index', y='Score', color='Metric', barmode='group',
             title='Precision, Recall, and F1-Score by Sentiment Class',
             labels={'index': 'Sentiment Class'})
fig.show()



📋 Classification Report:
              precision    recall  f1-score   support

  Irrelevant       0.77      0.29      0.42       172
    Negative       0.60      0.79      0.68       266
     Neutral       0.72      0.55      0.63       285
    Positive       0.60      0.79      0.69       277

    accuracy                           0.64      1000
   macro avg       0.67      0.61      0.60      1000
weighted avg       0.66      0.64      0.62      1000

